# Moustaghfir Ahmed Yassine - Bases du prompt engineering

appels LLM, modèles locaux, multimodal et extraction structurée.


In [ ]:
import os
import json
import base64
from pathlib import Path

import tiktoken
from IPython.display import Markdown, display, Image
from dotenv import load_dotenv

load_dotenv(override=True)


In [ ]:
tokenizer = tiktoken.encoding_for_model("gpt-4o")
prompt = "Vous êtes un assistant expert dans le domaine de l'analyse des sentiments."
tokens = tokenizer.encode(prompt)

print("Nombre de tokens :", len(tokens))
for token in tokens:
    print(tokenizer.decode_single_token_bytes(token).decode("utf-8", errors="ignore"), end=" | ")


In [ ]:
def tokens_count(prompt: str, model_encoding: str = "o200k_base") -> int:
    tokenizer = tiktoken.get_encoding(model_encoding)
    return len(tokenizer.encode(prompt))

tokens_count("Vous êtes un assistant.")


## Appel optionnel à OpenAI

Définir `OPENAI_API_KEY` dans `.env` avant d'exécuter cette cellule.


In [ ]:
if os.getenv("OPENAI_API_KEY"):
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import SystemMessage, HumanMessage

    llm_openai = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    response = llm_openai.invoke([
        SystemMessage(content="Vous êtes un assistant utile."),
        HumanMessage(content="C'est quoi un agent IA ?"),
    ])
    display(Markdown(response.content))
else:
    print("OPENAI_API_KEY absente : cellule ignorée.")


## Appel optionnel à Ollama

Nécessite Ollama lancé localement et un modèle installé, par exemple `llama3.2`.


In [ ]:
try:
    from langchain_ollama import ChatOllama
    from langchain_core.messages import HumanMessage

    llm_ollama = ChatOllama(model="llama3.2", temperature=0)
    response = llm_ollama.invoke([HumanMessage(content="C'est quoi un agent IA ?")])
    display(Markdown(response.content))
except Exception as exc:
    print("Ollama non disponible :", exc)


## Appel optionnel à Groq

Définir `GROQ_API_KEY` dans `.env` avant d'exécuter cette cellule.


In [ ]:
if os.getenv("GROQ_API_KEY"):
    from langchain_groq import ChatGroq
    from langchain_core.messages import HumanMessage

    llm_groq = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
    response = llm_groq.invoke([HumanMessage(content="C'est quoi un agent IA ?")])
    display(Markdown(response.content))
else:
    print("GROQ_API_KEY absente : cellule ignorée.")


## Exemple : extraction structurée de sentiments par aspect


In [ ]:
system_message = '''
Vous êtes un expert dans l'analyse des sentiments.
Effectuez une analyse de sentiment basée sur les aspects d'avis concernant des ordinateurs portables.
Les aspects sont : screen, keyboard, touchpad.
Pour chaque aspect, retournez une polarité : positive, negative ou neutral.
Répondez uniquement au format JSON avec les clés categories et polarities.
'''.strip()

avis = "J'ai beaucoup aimé l'écran. La souris n'est pas bonne et le clavier ma fih machef."

if os.getenv("OPENAI_API_KEY"):
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import SystemMessage, HumanMessage

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    response = llm.invoke([SystemMessage(content=system_message), HumanMessage(content=avis)])
    raw = response.content
else:
    raw = json.dumps({
        "categories": ["screen", "keyboard", "touchpad"],
        "polarities": ["positive", "negative", "neutral"],
    }, ensure_ascii=False)

result = json.loads(raw.replace("```json", "").replace("```", "").strip())
result
